In [ ]:
# Importing libraries
import os

import numpy as np
import pandas as pd

import ipywidgets as widgets
from IPython.display import display
from ipyfilechooser import FileChooser
from bokeh.plotting import figure, output_notebook, show
from bokeh.models import LabelSet, Label, ColumnDataSource

from qsarmodelingpy.external_validation import ExternalValidation
from qsarmodelingpy.cross_validation_class import CrossValidation
from qsarmodelingpy.kennardstonealgorithm import kennardstonealgorithm
import qsarmodelingpy.lj_cut as lj
from draw_widgets import DrawWidgets


dw = DrawWidgets()

## Run the cell below in order to generate the buttons to load the matrices 

In [ ]:
fileX = widgets.FileUpload(
    accept='',  # Accepted file extension e.g. '.txt', '.pdf', 'image/*', 'image/*,.pdf'
    multiple=False,  # True to accept multiple files upload else False
    description = "X matrix",
)
display(fileX)
filey = widgets.FileUpload(
    accept='',  # Accepted file extension e.g. '.txt', '.pdf', 'image/*', 'image/*,.pdf'
    multiple=False,  # True to accept multiple files upload else False
    description = "y vector",
)
display(filey)

## Run the cell below in order to adequate the matrices to the correct format used for calculations

In [ ]:
df = dw.mountMatrix(fileX)
y = dw.mountyvector(filey)

## Run the cell below to choose the external validation parameters. The values pre-selected are the optimal or default values.

In [ ]:
samples_widget = widgets.SelectMultiple(
    options=list(zip(df.index,range(len(df)))),
    #rows=10,
    description='Samples',
    disabled=False
)
samples_widget.layout.visibility = 'visible'

# TODO Kennard-Stone size

def show_samples(args):
    if args['new'] == 'Manual':
        samples_widget.layout.visibility = 'visible'
    else:
        samples_widget.layout.visibility = 'hidden'
        

# todo chamar validação cruzada e calcular nLV ótimo
nLVModel_widget = dw.drawIntSlider(value=int(len(df)/5),
                              min=1,
                              max=min(df.shape),
                              description='',
                              width="150pt"
                             )
display(widgets.HBox([widgets.Label('Maximum number of latent variables for the model:'), nLVModel_widget]))

autoscale_widget = widgets.RadioButtons(
    options=['Yes', 'No'],
    value='Yes',
    description='Autoscale?',
    disabled=False
)
display(autoscale_widget)

MIF_transform_widget = widgets.RadioButtons(
    options=['Yes', 'No'],
#     value='pineapple',
    description='',
    disabled=False,
    width="350pt"
)
widgets.HBox([widgets.Label('Transform MIF values?'), MIF_transform_widget])

method_widget = widgets.RadioButtons(
    options=['Manual', 'Kennard-Stone', 'Random'],
    value='Manual',
    description='Choose the external validation method',
    disabled=False
)

method_widget.observe(show_samples,'value')
display(method_widget)
display(samples_widget)

print("Choose the directory and type the filename to save the external validation results:")
fc_ext_val = FileChooser(os.getcwd())
display(fc_ext_val)

print("Choose the directory and type the filename to save the cross-validation results:")
fc_cv = FileChooser(os.getcwd())
display(fc_cv)

print("Choose the directory and type the filename to save the X matrix with the training set:")
fc_Xtrain = FileChooser(os.getcwd())
display(fc_Xtrain)

print("Choose the directory and type the filename to save the y vector with the training set:")
fc_ytrain = FileChooser(os.getcwd())
display(fc_ytrain)

print("Choose the directory and type the filename to save the X matrix with the test set:")
fc_Xtest = FileChooser(os.getcwd())
display(fc_Xtrain)

print("Choose the directory and type the filename to save the y vector with the test set:")
fc_ytest = FileChooser(os.getcwd())
display(fc_ytest)

In [ ]:
print(samples_widget.value)

In [ ]:
# Open configuration file in order to look for the matrices and the parameters to run
# external validation
nLV = nLVModel_widget.value
ext_val_file = fc_ext_val.selected
cv_file = fc_cv.selected
Xtrain_file = fc_Xtrain.selected
ytrain_file = fc_ytrain.selected
Xtest_file = fc_Xtest.selected
ytest_file = fc_ytest.selected
autoscale = autoscale_widget.value == "Yes"
dfX = MIF_transform_widget.value == "Yes" else dfX
X = dfX.values
type_ext_val = method_widget.value
if type_ext_val == "Manual": # manual selection
    test = samples_widget.value
    train = [j for j in range(len(y)) if j not in test]
elif type_ext_val == "Kennard-Stone": # Kennard-Stone
    size_test_set = int(dfConf[1][3])
    train,test = kennardstonealgorithm(dfX,len(dfX)-size_test_set) # parameter is the size of training set
else: # Random selection    
    pass
ext = ExternalValidation(X,y,nLV)
ext.extVal(train,test,nLV)
ext.saveExtVal(train,test,out_directory+"/"+ext_val_file)
cv = CrossValidation(X[train,:],y[train],nLVMax = nLV,scale=True)
cv.saveParameters(out_directory+"/"+cv_file)
dfXtrain = dfX.loc[dfX.index[train],dfX.columns]
dfXtrain.to_csv(Xtrain_file, sep =';')
dfytrain = pd.DataFrame(y[train])
dfytrain.to_csv(ytrain_file, sep =',', header=False)
dfXtest = dfX.loc[dfX.index[test],dfX.columns]
dfXtest.to_csv(Xtest_file, sep =';')
dfytest = pd.DataFrame(y[test])
dfytest.to_csv(ytest_file, sep =',', header=False)

In [ ]:
# Plot experimental X predited values of y for calibration
output_notebook()

source = ColumnDataSource(data=dict(
    y=y[test],
    y_pred=ext.ypred,
))

TOOLTIPS = [
    ("y", "@y"),
    ("y_pred", "@y_pred")
]

p = figure(title="External validation prediction", x_axis_label='Experimental pIC50', y_axis_label='Predicted pIC50',
          tooltips = TOOLTIPS)
    #p.text(x+0.3,y+0.3,flav.index[i])
p.circle("y","y_pred", source=source)
p.line(y[:,0],y[:,0],color="red")
show(p)